# Notebook 12 — ViT-Based Semantic Segmentation

## 🎯 Learning Objectives
* Adapt a Vision Transformer (ViT) for dense, pixel-wise prediction.
* Understand how image patches become Transformer tokens.
* Decode Transformer patch tokens back into a full-resolution segmentation mask.
* Compare ViT-Segmenter with U-Net on the same `SyntheticShapes` dataset.
* Analyze the trade-off between global context from self-attention and sharp local boundaries from U-Net-style skip connections.

---

## 🖼️ Big Picture
A classification ViT predicts one label for the whole image. Semantic segmentation is different: it predicts one class label for every pixel.

In this notebook, a `128×128` RGB image is split into `8×8` patches. Each patch becomes a token. A Transformer encoder processes all patch tokens with self-attention. The encoded tokens are then reshaped into a `16×16` feature map, and a pixel decoder upsamples that feature map back into a full `128×128` segmentation mask.

We do **not** use a `CLS` token here. A `CLS` token is useful when the model needs one image-level prediction. Segmentation needs dense spatial output, so we keep and decode the patch tokens instead.

---

## 📊 Dataset: SyntheticShapes
We use the same segmentation dataset as Notebook 11.

* **Input image:** `[B, 3, 128, 128]`
* **Target mask:** `[B, 128, 128]`
* **Classes:** `background`, `circle`, `square`, `triangle`
* **Objective:** assign one of the four classes to every pixel.

The model outputs pixel logits with shape:

```text
[B, 4, 128, 128]
```

For each pixel, the model produces four scores: one score per class.

---

## 🏗️ Architecture

```text
Input image
[B, 3, 128, 128]
        │
        ▼
PatchEmbedding, patch_size = 8
- 128 / 8 = 16 patches per side
- 16 × 16 = 256 patch tokens
- each patch has 8 × 8 × 3 = 192 values
- Linear(192 → 128)
        │
        ▼
Patch tokens
[B, 256, 128]
        │
        ▼
+ Sinusoidal Positional Encoding
[B, 256, 128]
        │
        ▼
TransformerEncoder × 4
- d_model = 128
- num_heads = 8
- head_dim = 128 / 8 = 16
- feed-forward network: 128 → 256 → 128
        │
        ▼
Encoded patch tokens
[B, 256, 128]
        │
        ▼
Reshape tokens into spatial feature map
[B, 256, 128] → [B, 128, 16, 16]
        │
        ▼
Pixel decoder

Stage 1:
BilinearUpsample ×2
Conv2d(128 → 32, k=3, pad=1)
BN + ReLU
Conv2d(32 → 32, k=3, pad=1)
BN + ReLU
→ [B, 32, 32, 32]

Stage 2:
BilinearUpsample ×2
Conv2d(32 → 32, k=3, pad=1)
BN + ReLU
Conv2d(32 → 32, k=3, pad=1)
BN + ReLU
→ [B, 32, 64, 64]

Stage 3:
BilinearUpsample ×2
Conv2d(32 → 16, k=3, pad=1)
BN + ReLU
Conv2d(16 → 16, k=3, pad=1)
BN + ReLU
→ [B, 16, 128, 128]

Final pixel classifier:
Conv2d(16 → 4, k=1)
        │
        ▼
Pixel logits
[B, 4, 128, 128]
```

### What each part does

**PatchEmbedding** turns image patches into vectors, similar to how word embeddings turn word IDs into vectors in NLP.

**Positional encoding** tells the Transformer where each patch came from in the image. Without this, the Transformer would know patch content but not patch location.

**TransformerEncoder** lets every patch attend to every other patch. This gives the model global context.

**Reshape** converts the token sequence back into a small 2D feature map.

**Pixel decoder** upsamples the `16×16` feature map back to `128×128`. Each stage doubles the spatial size using bilinear upsampling. The local `3×3` convolutions after each upsampling step help clean up pixel details.

---

## 💡 Theory: ViT-Segmenter vs U-Net

### Why use a Transformer here?

The Transformer encoder gives global context. Every patch can directly compare itself with every other patch.

For example, a patch near the top-left of the image can attend to a patch near the bottom-right in one layer. A CNN usually needs multiple layers before distant regions influence each other.

### Why does U-Net still have an advantage?

U-Net keeps high-resolution features through skip connections. These skip connections pass detailed spatial information from the encoder directly to the decoder.

This ViT-Segmenter does not use U-Net-style skip connections. It decodes only from the `16×16` patch-token grid. That keeps the model simple and clearly Transformer-based, but very sharp pixel boundaries are still harder than in U-Net.

### Patch size trade-off

Patch size controls the balance between spatial detail and attention cost.

* **Smaller patches:** more tokens, better spatial detail, higher memory and compute cost.
* **Larger patches:** fewer tokens, faster training, coarser masks.

Here:

```text
image_size = 128
patch_size = 8
128 / 8 = 16 patches per side
16 × 16 = 256 tokens
```

---

## 📉 Training Loss

Plain pixel-wise cross-entropy is not enough for this dataset because most pixels are background. A model can get high pixel accuracy while still doing poorly on small foreground shapes.

This notebook uses a combined segmentation loss:

### 1. Weighted Cross-Entropy

Handles class imbalance.

Background is common, so it receives a lower weight. Smaller foreground classes receive higher weights.

### 2. Foreground Dice Loss

Measures overlap between predicted shapes and target shapes.

The background class is ignored so the loss focuses on `circle`, `square`, and `triangle`.

### 3. Boundary Loss

Uses Sobel edges on predicted probabilities and target masks.

This encourages predicted shape boundaries to align better with target boundaries.

Final loss:

```text
loss = weighted_CE + 0.7 × foreground_Dice + 0.1 × boundary_loss
```

The goal is not only to classify pixels correctly, but also to improve foreground overlap and boundary quality.

## Implementation from Scratch
### 1. Imports and Setup

In [1]:
import sys
sys.path.insert(0, '..')  # Add repo root when running from notebooks/

import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import (
    save_metrics_json,
    save_markdown_report,
    update_runs_summary,
    save_table_csv,
)
from src.utils.paths import RUNS_SEGMENTATION_DIR
from src.data.synthetic_shapes import load_shapes_data
from src.models.vit_segmenter import ViTSegmenter as _BaseViTSegmenter
from src.metrics.segmentation import pixel_accuracy, mean_iou, dice_score
from src.visualization.segmentation import overlay_segmentation_mask, plot_segmentation_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_SEGMENTATION_DIR, clean=False)

print(f'Device  : {device}')
print(f'Run dir : {run_dir}')


def pixel_decoder_block(in_channels: int, out_channels: int) -> nn.Sequential:
    """Upsample by 2×, then refine local pixel details with two 3×3 convolutions."""
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),

        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


def build_pixel_decoder(
    d_model: int,
    num_classes: int,
    patch_size: int,
) -> nn.Sequential:
    """
    Build enough 2× upsampling blocks to recover the original image size.

    patch_size=16 → 4 blocks: 8×8  → 16×16 → 32×32 → 64×64 → 128×128
    patch_size=8  → 3 blocks: 16×16 → 32×32 → 64×64 → 128×128
    """
    num_upsample_blocks = int(math.log2(patch_size))

    if 2 ** num_upsample_blocks != patch_size:
        raise ValueError(f"patch_size must be a power of 2, got {patch_size}")

    channel_schedule = [32, 32, 16, 16]
    if num_upsample_blocks > len(channel_schedule):
        raise ValueError(
            f"patch_size={patch_size} needs {num_upsample_blocks} upsampling blocks, "
            f"but only {len(channel_schedule)} are defined."
        )

    layers = []
    in_channels = d_model

    for out_channels in channel_schedule[:num_upsample_blocks]:
        layers.append(pixel_decoder_block(in_channels, out_channels))
        in_channels = out_channels

    layers.append(nn.Conv2d(in_channels, num_classes, kernel_size=1))

    return nn.Sequential(*layers)


class ViTSegmenter(_BaseViTSegmenter):
    """
    Vision Transformer segmenter.

    The inherited part creates:
    image patches -> patch embeddings -> positional encoding -> Transformer encoder.

    This notebook uses a simple pixel decoder:
    bilinear upsample -> local 3×3 convolutions -> repeat until full resolution.
    """

    def __init__(
        self,
        image_size: int,
        patch_size: int,
        in_channels: int,
        num_classes: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float = 0.1,
    ):
        super().__init__(
            image_size=image_size,
            patch_size=patch_size,
            in_channels=in_channels,
            num_classes=num_classes,
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        self.decoder = build_pixel_decoder(
            d_model=d_model,
            num_classes=num_classes,
            patch_size=patch_size,
        )

[seed] Random seed set to 42.
Device  : cuda
Run dir : /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation


## Dataset
### 2. Load Synthetic Shapes (same as Notebook 11)

In [2]:
IMAGE_SIZE  = 128
PATCH_SIZE  = 8
NUM_CLASSES = 4

D_MODEL     = 128
NUM_HEADS   = 8
NUM_LAYERS  = 4
DIM_FF      = 256

BATCH_SIZE  = 16
NUM_EPOCHS  = 20
LR          = 6e-4

DICE_LOSS_WEIGHT     = 0.7
BOUNDARY_LOSS_WEIGHT = 0.1

CLASS_NAMES = ['background', 'circle', 'square', 'triangle']

print(f'Patch grid        : {IMAGE_SIZE // PATCH_SIZE}×{IMAGE_SIZE // PATCH_SIZE}')
print(f'Num patch tokens  : {(IMAGE_SIZE // PATCH_SIZE) ** 2}')
print(f'Head dimension    : {D_MODEL // NUM_HEADS}')

train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='segmentation',
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, masks = next(iter(train_loader))
print(f'Image batch shape : {imgs.shape}   (B, 3, H, W)')
print(f'Mask  batch shape : {masks.shape}  (B, H, W)')
print(f'Num patches       : {(IMAGE_SIZE//PATCH_SIZE)**2}  ({IMAGE_SIZE//PATCH_SIZE}×{IMAGE_SIZE//PATCH_SIZE} grid)')

def compute_class_weights(loader, num_classes, device):
    counts = torch.zeros(num_classes, dtype=torch.float64)

    for _, batch_masks in loader:
        counts += torch.bincount(
            batch_masks.reshape(-1),
            minlength=num_classes,
        ).double()

    freq = counts / counts.sum()
    weights = 1.0 / torch.sqrt(freq + 1e-8)
    weights = weights / weights.mean()
    weights = weights.clamp(max=5.0)

    print("\nClass pixel statistics:")
    for i, name in enumerate(CLASS_NAMES):
        print(
            f"  {i}: {name:<12} "
            f"pixels={int(counts[i]):>9}  "
            f"freq={freq[i].item():.4f}  "
            f"weight={weights[i].item():.3f}"
        )

    return weights.float().to(device)


def masks_to_one_hot(masks, num_classes):
    """Convert [B, H, W] integer masks to [B, C, H, W] one-hot masks."""
    return F.one_hot(masks, num_classes=num_classes).permute(0, 3, 1, 2).float()


def dice_loss(logits, targets, num_classes, ignore_background=True, eps=1e-6):
    probs = logits.softmax(dim=1)
    target_one_hot = masks_to_one_hot(targets, num_classes)

    intersection = (probs * target_one_hot).sum(dim=(0, 2, 3))
    denominator = probs.sum(dim=(0, 2, 3)) + target_one_hot.sum(dim=(0, 2, 3))

    dice_per_class = (2.0 * intersection + eps) / (denominator + eps)

    if ignore_background:
        dice_per_class = dice_per_class[1:]

    return 1.0 - dice_per_class.mean()


def sobel_edges(x):
    """Compute soft edge magnitude for [B, C, H, W] tensors."""
    channels = x.size(1)

    sobel_x = torch.tensor(
        [[-1, 0, 1],
         [-2, 0, 2],
         [-1, 0, 1]],
        dtype=x.dtype,
        device=x.device,
    ).view(1, 1, 3, 3)

    sobel_y = torch.tensor(
        [[-1, -2, -1],
         [ 0,  0,  0],
         [ 1,  2,  1]],
        dtype=x.dtype,
        device=x.device,
    ).view(1, 1, 3, 3)

    sobel_x = sobel_x.repeat(channels, 1, 1, 1)
    sobel_y = sobel_y.repeat(channels, 1, 1, 1)

    dx = F.conv2d(x, sobel_x, padding=1, groups=channels)
    dy = F.conv2d(x, sobel_y, padding=1, groups=channels)

    return torch.sqrt(dx.pow(2) + dy.pow(2) + 1e-6)


def boundary_loss(logits, targets, num_classes, ignore_background=True):
    probs = logits.softmax(dim=1)
    target_one_hot = masks_to_one_hot(targets, num_classes)

    if ignore_background:
        probs = probs[:, 1:]
        target_one_hot = target_one_hot[:, 1:]

    pred_edges = sobel_edges(probs)
    true_edges = sobel_edges(target_one_hot)

    return F.l1_loss(pred_edges, true_edges)


class_weights = compute_class_weights(
    train_loader,
    num_classes=NUM_CLASSES,
    device=device,
)


def segmentation_loss(logits, targets):
    ce = F.cross_entropy(logits, targets, weight=class_weights)

    dice = dice_loss(
        logits,
        targets,
        num_classes=NUM_CLASSES,
        ignore_background=True,
    )

    boundary = boundary_loss(
        logits,
        targets,
        num_classes=NUM_CLASSES,
        ignore_background=True,
    )

    return ce + DICE_LOSS_WEIGHT * dice + BOUNDARY_LOSS_WEIGHT * boundary

Patch grid        : 16×16
Num patch tokens  : 256
Head dimension    : 16
Image batch shape : torch.Size([16, 3, 128, 128])   (B, 3, H, W)
Mask  batch shape : torch.Size([16, 128, 128])  (B, H, W)
Num patches       : 256  (16×16 grid)

Class pixel statistics:
  0: background   pixels= 11051571  freq=0.8432  weight=0.299
  1: circle       pixels=   676469  freq=0.0516  weight=1.209
  2: square       pixels=   906134  freq=0.0691  weight=1.045
  3: triangle     pixels=   473026  freq=0.0361  weight=1.446


### 3. Build ViTSegmenter

In [3]:
model = ViTSegmenter(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=3,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=0.1,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters: {n_params:,}')

# Shape trace
x_sample = imgs[:2].to(device)          # [2, 3, 128, 128]
with torch.no_grad():
    logits = model(x_sample)             # [2, 4, 128, 128]

print(f'Input  shape : {x_sample.shape}')
print(f'Output shape : {logits.shape}  (same spatial size as input — check!)')
print(f'Num patches  : {model.num_patches}  ({model.num_patches_per_side}×{model.num_patches_per_side})')
assert logits.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), logits.shape
print('Decoder output shape check passed.')

Trainable parameters: 624,644
Input  shape : torch.Size([2, 3, 128, 128])
Output shape : torch.Size([2, 4, 128, 128])  (same spatial size as input — check!)
Num patches  : 256  (16×16)
Decoder output shape check passed.


## Sanity Checks

In [4]:
assert logits.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), f'Shape mismatch: {logits.shape}'
print('Output shape check passed.')

assert not torch.isnan(logits).any()
print('No NaN in output.')

preds = logits.argmax(dim=1)
masks_dev = masks[:2].to(device)
pa = pixel_accuracy(preds, masks_dev)
print(f'Random-init pixel accuracy : {pa:.3f}  (not expected to be exactly 0.25 because background dominates)')

loss_val = segmentation_loss(logits, masks_dev)
print(f'Initial segmentation loss : {loss_val.item():.4f}')
print('All sanity checks passed!')

Output shape check passed.
No NaN in output.
Random-init pixel accuracy : 0.101  (not expected to be exactly 0.25 because background dominates)
Initial segmentation loss : 2.1452
All sanity checks passed!


## Training

In [5]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)

history = {'train_loss': [], 'val_loss': [], 'pixel_accuracy': [], 'mean_iou': [], 'dice': []}

best_miou = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_masks in train_loader:
        batch_imgs  = batch_imgs.to(device)   # [B, 3, 128, 128]
        batch_masks = batch_masks.to(device)  # [B, 128, 128]
        optimizer.zero_grad()
        logits = model(batch_imgs)             # [B, 4, 128, 128]
        loss = segmentation_loss(logits, batch_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_pa, v_miou, v_dice, v_n = 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_masks in val_loader:
            batch_imgs  = batch_imgs.to(device)
            batch_masks = batch_masks.to(device)
            logits = model(batch_imgs)
            preds  = logits.argmax(dim=1)
            v_loss += segmentation_loss(logits, batch_masks).item()
            v_pa   += pixel_accuracy(preds, batch_masks)
            v_miou += mean_iou(preds, batch_masks, num_classes=NUM_CLASSES)
            v_dice += dice_score(preds, batch_masks, num_classes=NUM_CLASSES)
            v_n += 1
    current_val_loss = v_loss / v_n
    current_pxacc = v_pa / v_n
    current_miou = v_miou / v_n
    current_dice = v_dice / v_n
    
    history['val_loss'].append(current_val_loss)
    history['pixel_accuracy'].append(current_pxacc)
    history['mean_iou'].append(current_miou)
    history['dice'].append(current_dice)
    
    if current_miou > best_miou:
        best_miou = current_miou
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
    
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}  '
          f'pxacc={history["pixel_accuracy"][-1]:.4f}  '
          f'mIoU={history["mean_iou"][-1]:.4f}  '
          f'dice={history["dice"][-1]:.4f}')

if best_state is not None:
    model.load_state_dict({
        k: v.to(device)
        for k, v in best_state.items()
    })

print(f'Training complete! Best epoch: {best_epoch} | best mIoU: {best_miou:.4f}')

Epoch 1/20  train=1.3480  val=1.1443  pxacc=0.9645  mIoU=0.8285  dice=0.9027
Epoch 2/20  train=1.0407  val=1.0162  pxacc=0.9835  mIoU=0.9037  dice=0.9478
Epoch 3/20  train=0.8871  val=0.8126  pxacc=0.9708  mIoU=0.8608  dice=0.9236
Epoch 4/20  train=0.7452  val=0.6747  pxacc=0.9823  mIoU=0.9087  dice=0.9515
Epoch 5/20  train=0.6280  val=0.5702  pxacc=0.9884  mIoU=0.9388  dice=0.9681
Epoch 6/20  train=0.5292  val=0.4860  pxacc=0.9866  mIoU=0.9229  dice=0.9590
Epoch 7/20  train=0.4362  val=0.4078  pxacc=0.9906  mIoU=0.9480  dice=0.9731
Epoch 8/20  train=0.3718  val=0.3449  pxacc=0.9927  mIoU=0.9552  dice=0.9768
Epoch 9/20  train=0.3236  val=0.3094  pxacc=0.9889  mIoU=0.9375  dice=0.9674
Epoch 10/20  train=0.2789  val=0.2621  pxacc=0.9922  mIoU=0.9525  dice=0.9752
Epoch 11/20  train=0.2400  val=0.2297  pxacc=0.9947  mIoU=0.9653  dice=0.9821
Epoch 12/20  train=0.2160  val=0.2172  pxacc=0.9926  mIoU=0.9541  dice=0.9763
Epoch 13/20  train=0.1944  val=0.1845  pxacc=0.9973  mIoU=0.9818  dice=0.

## What to Watch During Training

Pixel accuracy can be misleading in segmentation because most pixels are background.

The more useful metrics here are:

- **Mean IoU**: measures overlap between predicted and true regions for each class.
- **Dice Score**: measures foreground overlap and is especially useful when objects are smaller than the background.
- **Per-class IoU**: shows whether the model is failing on a specific shape such as circle or triangle.

A good model should not only have high pixel accuracy. It should also have strong IoU for circle, square, and triangle.

## Evaluation

In [6]:
# Final evaluation
model.eval()
total_pa, total_miou, total_dice, n = 0.0, 0.0, 0.0, 0
with torch.no_grad():
    for batch_imgs, batch_masks in val_loader:
        preds = model(batch_imgs.to(device)).argmax(dim=1)
        total_pa   += pixel_accuracy(preds, batch_masks.to(device))
        total_miou += mean_iou(preds, batch_masks.to(device), num_classes=NUM_CLASSES)
        total_dice += dice_score(preds, batch_masks.to(device), num_classes=NUM_CLASSES)
        n += 1

final_pa   = total_pa   / n
final_miou = total_miou / n
final_dice = total_dice / n

print(f'ViT-Segmenter Pixel Accuracy : {final_pa:.4f}')
print(f'ViT-Segmenter Mean IoU       : {final_miou:.4f}')
print(f'ViT-Segmenter Dice Score     : {final_dice:.4f}')

from src.metrics.segmentation import intersection_over_union

all_pred_list, all_mask_list = [], []

model.eval()
with torch.no_grad():
    for batch_imgs, batch_masks in val_loader:
        preds = model(batch_imgs.to(device)).argmax(dim=1).cpu()
        all_pred_list.append(preds)
        all_mask_list.append(batch_masks)

all_preds_cat = torch.cat(all_pred_list)
all_masks_cat = torch.cat(all_mask_list)

print('\nPer-class IoU:')
for c, name in enumerate(CLASS_NAMES):
    iou_c = intersection_over_union(all_preds_cat, all_masks_cat, c)
    print(f'  {name:12s}: IoU = {iou_c:.4f}')

vit_metrics = {
    'pixel_accuracy': float(final_pa),
    'mean_iou': float(final_miou),
    'dice_score': float(final_dice),
    'best_epoch': int(best_epoch),
    'best_miou_during_training': float(best_miou),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
    'patch_size': PATCH_SIZE,
    'd_model': D_MODEL,
    'num_heads': NUM_HEADS,
    'num_layers': NUM_LAYERS,
    'dim_feedforward': DIM_FF,
}
save_metrics_json(vit_metrics, run_dir / 'vit_segmenter_metrics.json')
print(f'Metrics saved.')

ViT-Segmenter Pixel Accuracy : 0.9983
ViT-Segmenter Mean IoU       : 0.9875
ViT-Segmenter Dice Score     : 0.9937

Per-class IoU:
  background  : IoU = 0.9984
  circle      : IoU = 0.9840
  square      : IoU = 0.9915
  triangle    : IoU = 0.9792
Metrics saved.


In [7]:
# Compare with U-Net results
unet_metrics_path = run_dir / 'unet_metrics.json'
comparison_rows = []

if unet_metrics_path.exists():
    with open(unet_metrics_path) as f:
        unet_m = json.load(f)
    print('=== U-Net vs ViT-Segmenter Comparison ===')
    print(f'{"Model":<20} {"Px Acc":>8} {"mIoU":>8} {"Dice":>8} {"Params":>10}')
    print('-' * 60)
    print(f'{"U-Net":<20} {unet_m["pixel_accuracy"]:>8.4f} {unet_m["mean_iou"]:>8.4f} '
          f'{unet_m["dice_score"]:>8.4f} {unet_m["num_params"]:>10,}')
    print(f'{"ViT-Segmenter":<20} {final_pa:>8.4f} {final_miou:>8.4f} '
          f'{final_dice:>8.4f} {n_params:>10,}')

    comparison_rows = [
        {'model': 'U-Net', 'pixel_accuracy': unet_m['pixel_accuracy'],
         'mean_iou': unet_m['mean_iou'], 'dice_score': unet_m['dice_score'],
         'num_params': unet_m['num_params'], 'epochs': unet_m['num_epochs']},
        {'model': 'ViT-Segmenter', 'pixel_accuracy': final_pa,
         'mean_iou': final_miou, 'dice_score': final_dice,
         'num_params': n_params, 'epochs': NUM_EPOCHS},
    ]
else:
    print('U-Net metrics not found — run Notebook 11 first.')
    comparison_rows = [
        {'model': 'ViT-Segmenter', 'pixel_accuracy': final_pa,
         'mean_iou': final_miou, 'dice_score': final_dice,
         'num_params': n_params, 'epochs': NUM_EPOCHS},
    ]

# Save comparison CSV
save_table_csv(comparison_rows, run_dir / 'vit_vs_unet_comparison.csv')
print(f'Comparison CSV saved to: {run_dir}/vit_vs_unet_comparison.csv')

=== U-Net vs ViT-Segmenter Comparison ===
Model                  Px Acc     mIoU     Dice     Params
------------------------------------------------------------
U-Net                  1.0000   0.9998   0.9999  1,948,388
ViT-Segmenter          0.9983   0.9875   0.9937    624,644
Comparison CSV saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_vs_unet_comparison.csv


## Visualization

In [8]:
# Training curves
from src.visualization.plots import plot_training_curves
plot_training_curves(
    {'train_loss': history['train_loss'], 'val_loss': history['val_loss'],
     'mean_iou': history['mean_iou'], 'dice': history['dice']},
    save_path=run_dir / 'vit_segmenter_training_curve.png',
    title='ViT-Segmenter Training Curves (Synthetic Shapes)',
)
print('Training curve saved.')

Training curve saved.


In [9]:
# Segmentation examples
model.eval()
vis_imgs, vis_masks = next(iter(val_loader))
with torch.no_grad():
    vis_preds = model(vis_imgs.to(device)).argmax(dim=1).cpu()

overlay_segmentation_mask(
    image=vis_imgs[0].numpy(),
    mask=vis_preds[0].numpy(),
    alpha=0.5,
    save_path=run_dir / 'vit_segmenter_overlay.png',
    title='ViT-Segmenter Predicted Segmentation Overlay',
    class_names=['background', 'circle', 'square', 'triangle'],
)

plot_segmentation_examples(
    images=vis_imgs.numpy(),
    gt_masks=vis_masks.numpy(),
    pred_masks=vis_preds.numpy(),
    save_path=run_dir / 'vit_segmenter_examples.png',
    title='ViT-Segmenter: Image | GT Mask | Predicted Mask',
    n_examples=4,
)
print('Visualisations saved.')

Visualisations saved.


In [10]:
# Side-by-side comparison bar chart
if len(comparison_rows) == 2:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metric_names = ['pixel_accuracy', 'mean_iou', 'dice_score']
    metric_labels = ['Pixel Accuracy', 'Mean IoU', 'Dice Score']
    model_labels = [r['model'] for r in comparison_rows]
    colors = ['steelblue', 'tomato']

    for ax, mname, mlabel in zip(axes, metric_names, metric_labels):
        vals = [r[mname] for r in comparison_rows]
        bars = ax.bar(model_labels, vals, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=10)
        ax.set_ylim(0, 1.1)
        ax.set_title(mlabel)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('U-Net vs ViT-Segmenter — Synthetic Shapes')
    plt.tight_layout()
    fig.savefig(run_dir / 'unet_vs_vit_metrics_comparison.png', dpi=120)
    plt.close(fig)
    print('Comparison chart saved.')

Comparison chart saved.


## Failure Cases

**Why ViT-Segmenter may still underperform U-Net on this dataset:**

1. **No high-resolution skip connections**  
   U-Net passes detailed encoder features directly to the decoder. ViT-Segmenter decodes only from the `16×16` patch-token grid, so exact pixel boundaries are still harder than in U-Net.

2. **Patch-token bottleneck**  
   With `patch_size=8`, the Transformer sees the image as 256 tokens. This gives more spatial detail than `patch_size=16`, but the model still reconstructs the final mask from patch tokens rather than high-resolution encoder features.

3. **Boundary artifacts**  
   Bilinear upsampling and `3×3` decoder convolutions reduce blockiness, but some stair-step or patch-like edges can still remain.

4. **Dataset bias toward CNNs**  
   Synthetic shapes have simple, sharp geometric boundaries. This strongly favors U-Net-style local processing.

5. **Parameter budget**  
   ViT-Segmenter still uses fewer parameters than U-Net in this notebook. The comparison therefore shows both accuracy and parameter efficiency.

**When ViT-style segmentation becomes stronger:**
- Large datasets.
- Complex scenes where global context matters.
- Pre-trained Transformer backbones.
- Hierarchical Transformer models such as Swin, SegFormer, or Mask2Former.

## Exercises

1. **Patch size ablation**  
   Change `PATCH_SIZE` from 16 to 8. This increases the token grid from 8×8 to 16×16. Does mean IoU improve? How much slower does training become?

2. **Decoder ablation**  
   Remove the 3×3 convolution after each `ConvTranspose2d` block. Does the mask become more blocky? Does mIoU drop?

3. **Loss ablation**  
   Train with only weighted cross-entropy. Then train with weighted cross-entropy + Dice loss. Finally train with the full loss including boundary loss. Which term helps foreground objects most?

4. **Attention map visualization**  
   Extract attention weights from the last Transformer encoder block. Pick one patch inside a shape and visualize which other patches it attends to.

5. **Layer depth ablation**  
   Train with `NUM_LAYERS=1`, `2`, and `4`. Does more Transformer depth improve mIoU, or does it overfit?

## Key Takeaways

- ViT-Segmenter converts an image into patch tokens, processes them with a Transformer encoder, reshapes them into a 2D feature grid, and decodes that grid into pixel-wise logits.
- No CLS token is used because segmentation needs one prediction per pixel.
- The Transformer encoder gives global context: every patch can attend to every other patch.
- The pixel decoder upsamples the 8×8 token grid back to 128×128 using `ConvTranspose2d` and local 3×3 convolutions.
- Weighted cross-entropy, foreground Dice loss, and boundary loss prevent training from being dominated by background pixels.
- U-Net still has an advantage in boundary precision because it uses high-resolution skip connections.
- ViT-Segmenter is much smaller than U-Net here, so the comparison shows the trade-off between parameter efficiency and pixel-perfect masks.
- The comparison CSV (`vit_vs_unet_comparison.csv`) is used in the final summary notebook.

## Saved Outputs

In [11]:
save_markdown_report(
    title='ViT-Segmenter — Semantic Segmentation',
    summary=(
        f'ViTSegmenter (image_size={IMAGE_SIZE}, patch_size={PATCH_SIZE}, '
        f'd_model={D_MODEL}, heads={NUM_HEADS}, layers={NUM_LAYERS}) '
        f'trained for {NUM_EPOCHS} epochs. '
        f'Best epoch: {best_epoch}. '
        f'Pixel acc: {final_pa:.4f}, mean IoU: {final_miou:.4f}, Dice: {final_dice:.4f}.'
    ),
    metrics=vit_metrics,
    figures=[
        'vit_segmenter_training_curve.png',
        'vit_segmenter_overlay.png',
        'vit_segmenter_examples.png',
    ],
    tables=['vit_vs_unet_comparison.csv'],
    output_path=run_dir / 'vit_segmenter_report.md',
    device=str(device),
    hyperparams={
        'patch_size': PATCH_SIZE,
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'dim_feedforward': DIM_FF,
        'batch_size': BATCH_SIZE,
        'epochs': NUM_EPOCHS,
        'lr': LR,
    },
    architecture=(
        f'PatchEmbedding({PATCH_SIZE}×{PATCH_SIZE}×3 → {D_MODEL}) + '
        f'SinusoidalPE + TransformerEncoder×{NUM_LAYERS} '
        f'({NUM_HEADS} heads, FFN {D_MODEL}→{DIM_FF}→{D_MODEL}) → '
        f'reshape [B, {D_MODEL}, {IMAGE_SIZE // PATCH_SIZE}, {IMAGE_SIZE // PATCH_SIZE}] → '
        f'PixelDecoder: '
        f'(BilinearUpsample×2 + Conv3×3 + BN + ReLU + Conv3×3 + BN + ReLU)×{int(math.log2(PATCH_SIZE))} → '
        f'Conv1×1({NUM_CLASSES} classes)'
    ),
    loss_fn=(
        f'Weighted CrossEntropyLoss + '
        f'{DICE_LOSS_WEIGHT} × foreground DiceLoss + '
        f'{BOUNDARY_LOSS_WEIGHT} × boundary loss'
    ),
)

update_runs_summary(
    session_name='vit_segmentation',
    report_path=run_dir / 'vit_segmenter_report.md',
    metrics=vit_metrics,
    figures=['vit_segmenter_training_curve.png', 'vit_segmenter_examples.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/vit_segmenter_metrics.json')
print(f'  {run_dir}/vit_segmenter_training_curve.png')
print(f'  {run_dir}/vit_segmenter_overlay.png')
print(f'  {run_dir}/vit_vs_unet_comparison.csv')
print(f'  {run_dir}/vit_segmenter_report.md')

All outputs saved:
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_metrics.json
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_training_curve.png
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_overlay.png
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_vs_unet_comparison.csv
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_report.md
